## Использование Gamac на реальных данных
Описание реальных задач приведено в [документации](https://github.com/ITMO-CODE-AI/GaMAC/blob/develop/docs/CASE_RU.md)

### Импортируем нужные библиотеки

In [2]:
import os
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
from PIL import Image

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### 1. Данные промышленных логов (семпл)
Кластеризация логов направлена на автоматическое выделение групп записей, связанных с общими источниками событий, такими как прикладные процессы, системные события или среды исполнения. Основная цель — обнаружить скрытые паттерны в данных, которые позволяют косвенно определить природу возникновения логов, даже если их формат не содержит явных указаний на источник.

Для получения данных нужно выгрузить их из Minio, данные лежат в: datasets/data/synthetic_logs_1kk.csv

И переложить из корня в папку data/

Примечание: выполнение по этому датасету проходит очень долгое время в связи с его объемом

In [7]:
# Импортируем датафрейм
data = pd.read_csv('../data/synthetic_logs_1kk.csv')

In [8]:
data.head(5)

,object,agent,subject,environment,source,result,result code,tag,log level,label,traceback,protocol,action,property,timestamp
0,missing,missing,thread,missing,24.149.203.18,[<:subject:> <:source:>] File does not exist:<...,missing,missing,error,missing,event interval expired,missing,missing,lifetime 05:00,2025-04-18 10:08:58
1,dfs.FSNamesystem,ocsp.digicert.com:80,out block blk 8855628190418004805,missing,transfer block blk -1132082380757283113 to 10....,<:action:>* NameSystem.addStoredBlock: blockMa...,missing,27052.0,WARN,missing,Redundant addStoredBlock request received for ...,missing,Access denied,power/control problem,2025-03-28 10:08:58
2,spoolsv.exe *64,news.17173.com:80,thread,missing,transfer block blk -1009207079038502874 to 10....,link errors remain current,missing,6708.0,-1,state_change.unavailable,Redundant addStoredBlock request received for ...,missing,missing,power/control problem,2025-04-19 10:08:58
3,chrome.exe,pubads.g.doubleclick.net:443,node-139,missing,84.92.64.33,"<:action:>, <:NUM:> bytes <:*:> <:*:> sent, <:...",missing,missing,missing,missing,timed,missing,close,power/control problem,2025-04-14 10:08:58
4,partition,mini.cpc.sogou.com:80,thread,missing,85.64.10.104,critical,missing,385417.0,-1,status,must start with /,missing,missing,power/control problem,2025-04-16 10:08:58


In [5]:
# По умолчанию кол-во итераций стоит 50
gamac = Gamac(iter_limit=100, target_measures={Internal.BR})

dataset, best_model = gamac.run(table=data)

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.06452703475952148s, {'BR': -15.240080401029442} ===
=== MEASURES 0.002025604248046875s, {'BR': -13.793766265861501} ===
=== ALGO 0.11502766609191895s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 3, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 34.18908905982971s, FailedRun, {'bandwidth': 0.5976054974793465, 'max_iter': 270, 'tol': 4.250531313551204e-05} ===
=== ALGO 2.977846622467041s, FailedRun, {'name': 'DBSCAN', 'eps': 0.5434155909882943, 'eps_sq': 0.2953005045291571, 'min_samples': 11} ===
=== ALGO 2.114518642425537s, FailedRun, {'min_cluster_size': 13, 'min_samples': 5} ===
=== MEASURES 0.0021827220916748047s, {'BR': -14.1702662908328} ===
=== ALGO 63.865172386169434s, SuccessRun, {'threshold': 0.17742247629602081, 'branching_factor': 62, 'n_clusters': 2} ===
=== MEASURES 0.0036377906799316406s, {'BR': -12.139171364162541} ===
=== ALGO 0.07503128051757812s, SuccessRun, {'name': 'KMeans', 'n_clusters': 8, 'max_iter': 100, 'tol': 0.0001, 

In [6]:
# Получение топ-50 меток по лучшему классификатору
print(best_model.model.predict(dataset)[:50])

[11  0  9 13  5  0  0  6 13 14  3  8  1 12  7 14  8 10 14 13  3 12  9 14
  9  0  2 13 11  7  6 12 13  9 14 12  8  0 14  5 10 11 10  9  9 12 10  8
  0 14]


### 2. Данные по анализу цветов RGB-изображений
Кластеризация цветов RGB-изображений направлена на автоматическое группирование пикселей со схожими цветовыми значениями в отдельные кластеры, где каждый кластер представляет собой усреднённый цвет или доминирующий оттенок из соответствующей группы. Основная цель — упростить представление изображения за счёт выделения ключевых цветовых паттернов, что может быть полезно для задач сжатия данных, сегментации объектов, снижения шума или анализа цветовой палитры. 

In [2]:
# Импортируем картинки
images = []

# Получаем список файлов в директории
for filename in os.listdir('../data/images/'):
    if 'png' not in filename:
        continue
    file_path = os.path.join('../data/images/', filename)

    # Пытаемся открыть файл как изображение
    with Image.open(file_path) as img:
        images.append(img.copy())

images[:5]

[<PIL.Image.Image image mode=RGBA size=424x628>,
 <PIL.Image.Image image mode=RGBA size=802x545>,
 <PIL.Image.Image image mode=RGBA size=391x504>,
 <PIL.Image.Image image mode=RGBA size=400x570>,
 <PIL.Image.Image image mode=RGBA size=450x555>]

### Запустим подбор

In [3]:
# Для кластеризации передадим константный пустой текст
gamac = Gamac(iter_limit=30)
empty_texts = ['' for i in range(len(images))]

dataset, best_model = gamac.run(image=images, text=empty_texts)

/home/user/miniconda3/envs/GaMAC-org/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== Started CVI prediction ===
=== CVI prediction iteration 1/2 ====
=== CVI prediction iteration 2/2 ====
=== Picked MCR in 10.756967067718506s ===
=== MEASURES 0.13705706596374512s, {'MCR': -2.9950725765136115} ===
=== MEASURES 0.09331893920898438s, {'MCR': -2.3441037167739114} ===
=== ALGO 0.36558985710144043s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 4, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 0.9220912456512451s, FailedRun, {'bandwidth': 0.06536588142902727, 'max_iter': 101, 'tol': 3.450727222256216e-05} ===
=== ALGO 0.758134126663208s, FailedRun, {'name': 'DBSCAN', 'eps': 0.7562775366070154, 'eps_sq': 0.5719557123763754, 'min_samples': 15} ===
=== ALGO 0.2525618076324463s, FailedRun, {'min_cluster_size': 12, 'min_samples': 10} ===
=== MEASURES 0.12599921226501465s, {'MCR': -198.90139954374936} ===
=== ALGO 0.573908805847168s, SuccessRun, {'threshold': 0.8828107170979665, 'branching_factor': 37, 'n_clusters': 9} ===
=== MEASURES 0.112586975097656

In [4]:
# Метки по датасету
print(best_model.model.predict(dataset))

[2 0 5 2 2 1 0 2 2 2 4 1 2 5 2 3 2 5 5 1 2 1 4 5 2 2 5 5 2 5 5 1 1 2 5 5 2
 5 5 5 5 2 5 2 3 1 5 2 5 5 5 4 5 2 5 5 5 5 2 5 5 3 2 5 2 5 2 5 5 2 4 1 3 5
 5 4 5 2 2 2 1 1 5 5 5 2 1 1 5 1 1 5 5 2 5 5 1 5 5 5 5 2 5 1 5 3 2 2 5 2 2
 4 5 1 1 4 4 2 5 5 5 5 5 5 2 5 1 2 3 2 1 4 2 1 5 2 1 1 2 5 2 5 5 2 5 2 2 2
 4 5 5 2 2 1 5 5 5 5 5 1 2 5 0 4 1 2 2 2 3 3 2 5 5 5 3 5 2 5 5 5 0 5 2 0 5
 1 2 2 2 2 5 5 1 5 5 3 5 5 1 2 5 2 2 3 5 2 5 2 5 5 5 2 2 1 2 5 2 2 5 2 5 1
 5 2 5 2 5 2 2 4 1 5 5 5 5 3 2 4 1 2 2 4 5 5 3 5 2 2 2 5 1 5 1 5 2 2 2 2 5
 4 2 3 1 2 0 2 1 5 1 0 5 5 2 5 1 4 5 5 3 2 2 2 5 1 3 5 1 2 5 5 5 2 5 5 2 2
 2 5 5 2 5 5 5 2 2 3 3 2 2 2 5 5 5 4 1 2 1 5 2 2 2 5 2 2 2 1 1 2 4 5 1 5 5
 3 2 2 0 5 2 5 5 5 3 5 1 5 5 2 1 2 4 0 2 2 4 5 5 5 5 4 5 5 5 5 1 2 5 2 5 5
 2 5 2 2 5 2 2 1 4 5 2 5 2 1 5 3 3 4 5 2 5 2 3 3 4 5 5 1 1 1 5 5 3 5 5 5 2
 5 5 2 3 5 5 2 1 2 5 5 5 2 2 5 1 5 0 5 5 2 5 2 5 5 5 2 2 1 5 2 5 3 2 2 5 4
 2 5 2 1 2 5 5 3 1 1 2 5 5 5 1 5 3 5 0 5 4 5 5 2 2 5 2 1 2 5 5 2 1 3 1 5 2
 4 2 3 2 2 5 5 2 5 3 5 5 